# Comparison file
**For comparing the custom implementations to the sklearn one**

In [4]:
from customRegressionTreeForest import RegressionTreeNico, RandomForestNico

# Other imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional, Tuple, Dict, Any, List

# for testing and comparison
import time
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [6]:
def load_dataset(csv_path: Optional[str] = None, target_col: Optional[str] = None, random_state: int = 42
                ) -> Tuple[pd.DataFrame, str]:
    """
    Load dataset from CSV or fall back to California housing or synthetic regression.
    Returns (df, target_name).
    """
    if csv_path:
        df = pd.read_csv(csv_path)
        if target_col is None:
            target_col = df.columns[-1]
        return df, target_col

    # Try California housing (realistic regression) else synthetic
    try:
        data = fetch_california_housing(as_frame=True)
        df = data.frame.copy()
        # sklearn's fetch returns target in data.target; ensure a column name
        if "target" not in df.columns:
            df["target"] = data.target
            target_col = "target"
        else:
            target_col = df.columns[-1]
        return df, target_col
    except Exception:
        X, y = make_regression(n_samples=5000, n_features=10, noise=0.1, random_state=random_state)
        df = pd.DataFrame(X, columns=[f"X{i}" for i in range(X.shape[1])])
        df["target"] = y
        return df, "target"

def split_df(df: pd.DataFrame, target_name: str, test_size: float = 0.2, random_state: int = 42
            ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    X = df.drop(columns=[target_name])
    y = df[target_name]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    return X_train, X_test, y_train, y_test

def metrics_dict(y_true, y_pred) -> Dict[str, float]:
    return {
        "MSE": float(mean_squared_error(y_true, y_pred)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred))
    }

# scikit-learn pipeline
def sklearn_pipeline(X_train: pd.DataFrame, y_train: pd.Series, X_test: pd.DataFrame, y_test: pd.Series,
                     random_state: int = 42) -> Dict[str, Any]:
    out = {}
    clf = DecisionTreeRegressor(random_state=random_state)
    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred = clf.predict(X_test)
    pred_time = time.perf_counter() - t0

    out["sklearn_unpruned"] = {
        "model": clf,
        "fit_time": fit_time,
        "predict_time": pred_time,
        "metrics": metrics_dict(y_test, y_pred)
    }

    # cost complexity pruning path (on training data)
    t0 = time.perf_counter()
    path = clf.cost_complexity_pruning_path(X_train, y_train)
    path_time = time.perf_counter() - t0
    out["sklearn_ccp_path"] = {
        "ccp_alphas": path.ccp_alphas,
        "impurities": path.impurities,
        "path_time": path_time
    }

    # train one pruned model for demonstration (choose a middle non-zero alpha if available)
    ccp_alphas = path.ccp_alphas
    if ccp_alphas.size > 1:
        nonzero = ccp_alphas[ccp_alphas > 0]
        chosen_alpha = float(nonzero[len(nonzero)//2]) if len(nonzero) > 0 else float(ccp_alphas[-1])
        clf_pruned = DecisionTreeRegressor(random_state=random_state, ccp_alpha=chosen_alpha)
        t0 = time.perf_counter()
        clf_pruned.fit(X_train, y_train)
        pruned_fit_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        y_pred_pruned = clf_pruned.predict(X_test)
        pruned_pred_time = time.perf_counter() - t0

        out["sklearn_pruned"] = {
            "alpha": chosen_alpha,
            "model": clf_pruned,
            "fit_time": pruned_fit_time,
            "predict_time": pruned_pred_time,
            "metrics": metrics_dict(y_test, y_pred_pruned)
        }
    else:
        out["sklearn_pruned"] = None

    return out

def custom_pipeline(CustomTreeClass, df_train: pd.DataFrame, target_name: str,
                    X_test: pd.DataFrame, y_test: pd.Series,
                    alpha_for_pruning: Optional[float] = None, random_state: int = 42,
                    categorical_features: List[str] = []) -> Dict[str, Any]:
    out = {}
    model = CustomTreeClass(random_state=random_state)

    # fit without pruning
    t0 = time.perf_counter()
    model.fit(df_train, target_name, alpha=0.0, categorical_features=categorical_features)
    fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred = model.predict(X_test)
    pred_time = time.perf_counter() - t0

    out["custom_unpruned"] = {
        "model": model,
        "fit_time": fit_time,
        "predict_time": pred_time,
        "metrics": metrics_dict(y_test, y_pred)
    }

    # compute pruning path if available
    try:
        t0 = time.perf_counter()
        path = model.cost_complexity_pruning_path(model.tree_)
        path_time = time.perf_counter() - t0
        out["custom_ccp_path"] = {"path": path, "path_time": path_time}
    except Exception as e:
        out["custom_ccp_path"] = {"error": str(e)}

    # fit with pruning if requested
    if alpha_for_pruning is not None and alpha_for_pruning > 0.0:
        t0 = time.perf_counter()
        model_pruned = CustomTreeClass(random_state=random_state)
        model_pruned.fit(df_train, target_name, alpha=alpha_for_pruning, categorical_features=categorical_features)
        prune_fit_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        y_pred_pruned = model_pruned.predict(X_test)
        prune_pred_time = time.perf_counter() - t0

        out["custom_pruned"] = {
            "alpha": alpha_for_pruning,
            "model": model_pruned,
            "fit_time": prune_fit_time,
            "predict_time": prune_pred_time,
            "metrics": metrics_dict(y_test, y_pred_pruned)
        }
    else:
        out["custom_pruned"] = None

    return out

# -------------------------
# runner and reporting
# -------------------------
def compare_and_report(sk_results: Dict[str, Any], custom_results: Optional[Dict[str, Any]] = None,
                       show_pruning_path_entries: int = 10):
    print("\n=== scikit-learn: unpruned ===")
    su = sk_results["sklearn_unpruned"]
    print(f"fit_time: {su['fit_time']:.4f}s, predict_time: {su['predict_time']:.4f}s")
    print("metrics:", su["metrics"])

    if sk_results.get("sklearn_pruned"):
        sp = sk_results["sklearn_pruned"]
        print("\n=== scikit-learn: pruned (example) ===")
        print(f"alpha: {sp['alpha']}, fit_time: {sp['fit_time']:.4f}s, predict_time: {sp['predict_time']:.4f}s")
        print("metrics:", sp["metrics"])

    print("\nscikit-learn pruning path (first entries):")
    ccp = sk_results["sklearn_ccp_path"]
    for a, imp in list(zip(ccp["ccp_alphas"], ccp["impurities"]))[:show_pruning_path_entries]:
        print(f"  alpha={a:.6g}, impurity={imp:.6g}")
    print(f"  (path computation time: {ccp['path_time']:.4f}s)")

    if custom_results is None:
        print("\nCustom implementation not provided or not importable.")
        return

    print("\n=== Custom tree: unpruned ===")
    cu = custom_results["custom_unpruned"]
    print(f"fit_time: {cu['fit_time']:.4f}s, predict_time: {cu['predict_time']:.4f}s")
    print("metrics:", cu["metrics"])

    if custom_results.get("custom_pruned"):
        cp = custom_results["custom_pruned"]
        print("\n=== Custom tree: pruned ===")
        print(f"alpha: {cp['alpha']}, fit_time: {cp['fit_time']:.4f}s, predict_time: {cp['predict_time']:.4f}s")
        print("metrics:", cp["metrics"])

    print("\nCustom pruning path (first entries):")
    cpath = custom_results.get("custom_ccp_path", {})
    if "path" in cpath and isinstance(cpath["path"], list):
        for entry in cpath["path"][:show_pruning_path_entries]:
            # entry expected as (effective_alpha, num_leaves, subtree_error)
            print(f"  eff_alpha={entry[0]:.6g}, num_leaves={entry[1]}, subtree_err={entry[2]:.6g}")
        print(f"  (path computation time: {cpath['path_time']:.4f}s)")
    else:
        print("  (no path or error):", cpath.get("error"))

    # compact comparison table
    rows = []
    rows.append({
        "model": "sklearn_unpruned",
        **su["metrics"],
        "fit_time": su["fit_time"],
        "predict_time": su["predict_time"]
    })
    if sk_results.get("sklearn_pruned"):
        rows.append({
            "model": f"sklearn_pruned_alpha={sk_results['sklearn_pruned']['alpha']}",
            **sk_results['sklearn_pruned']["metrics"],
            "fit_time": sk_results['sklearn_pruned']["fit_time"],
            "predict_time": sk_results['sklearn_pruned']["predict_time"]
        })
    rows.append({
        "model": "custom_unpruned",
        **cu["metrics"],
        "fit_time": cu["fit_time"],
        "predict_time": cu["predict_time"]
    })
    if custom_results.get("custom_pruned"):
        rows.append({
            "model": f"custom_pruned_alpha={custom_results['custom_pruned']['alpha']}",
            **custom_results['custom_pruned']["metrics"],
            "fit_time": custom_results['custom_pruned']["fit_time"],
            "predict_time": custom_results['custom_pruned']["predict_time"]
        })

    df_comp = pd.DataFrame(rows).set_index("model")
    print("\n=== Summary comparison ===")
    print(df_comp.round(6).to_string())

# -------------------------
# main runner
# -------------------------
def main(csv_path: Optional[str] = None, target_col: Optional[str] = None,
         random_state: int = 42, test_size: float = 0.2, alpha_choice: Optional[float] = None,
         evaluate_alphas: Optional[List[float]] = None, save_csv: Optional[str] = None):
    df, target_name = load_dataset(csv_path, target_col, random_state=random_state)
    print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} cols. Target: {target_name}")

    X_train, X_test, y_train, y_test = split_df(df, target_name, test_size=test_size, random_state=random_state)
    df_train = pd.concat([X_train, y_train], axis=1)

    print("\nRunning scikit-learn pipeline...")
    sk_results = sklearn_pipeline(X_train, y_train, X_test, y_test, random_state=random_state)

    # choose alpha for custom if not provided
    chosen_alpha = alpha_choice
    if chosen_alpha is None:
        ccp_alphas = sk_results["sklearn_ccp_path"]["ccp_alphas"]
        nonzero = ccp_alphas[ccp_alphas > 0]
        if len(nonzero) > 0:
            chosen_alpha = float(nonzero[len(nonzero)//2])
        else:
            chosen_alpha = float(ccp_alphas[-1]) if ccp_alphas.size > 0 else 0.0

    custom_results = None
    if RegressionTreeNico is not None:
        print("\nRunning custom tree pipeline...")
        custom_results = custom_pipeline(RegressionTreeNico, df_train, target_name, X_test, y_test,
                                         alpha_for_pruning=chosen_alpha, random_state=random_state)
    else:
        print("\nRegressionTreeNico class not found. Please ensure it's importable or defined in the same file.")

    compare_and_report(sk_results, custom_results)

    # optional: evaluate multiple alphas for the custom tree and save CSV
    if RegressionTreeNico is not None and evaluate_alphas:
        records = []
        for a in evaluate_alphas:
            print(f"\nEvaluating custom tree with alpha={a}")
            res = custom_pipeline(RegressionTreeNico, df_train, target_name, X_test, y_test,
                                  alpha_for_pruning=a, random_state=random_state)
            pr = res.get("custom_pruned") or res.get("custom_unpruned")
            records.append({
                "alpha": a,
                **pr["metrics"],
                "fit_time": pr["fit_time"],
                "predict_time": pr["predict_time"]
            })
        df_alphas = pd.DataFrame(records).set_index("alpha").sort_index()
        print("\nMetrics vs alpha (custom tree):")
        print(df_alphas.round(6).to_string())
        if save_csv:
            df_alphas.to_csv(save_csv)
            print(f"Saved alpha results to {save_csv}")

    return {"sklearn": sk_results, "custom": custom_results}

# If run as script
if __name__ == "__main__":
    import sys
    csv = None
    target = None
    if len(sys.argv) >= 2:
        csv = sys.argv[1]
    if len(sys.argv) >= 3:
        target = sys.argv[2]
    # Example: evaluate a small grid of alphas for the custom tree and save to file
    main(target_col=target, evaluate_alphas=[0.0, 1e-6, 1e-5, 1e-4, 1e-3], save_csv="alpha_results.csv")

Dataset: 20640 rows, 10 cols. Target: target

Running scikit-learn pipeline...

Running custom tree pipeline...

=== scikit-learn: unpruned ===
fit_time: 0.1125s, predict_time: 0.0018s
metrics: {'MSE': 3.928779118217063e-06, 'MAE': 0.00025896802325959324, 'R2': 0.999997001867979}

=== scikit-learn: pruned (example) ===
alpha: 1.6771019663996794e-10, fit_time: 0.1306s, predict_time: 0.0010s
metrics: {'MSE': 4.014133372660915e-06, 'MAE': 0.0004313834502810362, 'R2': 0.9999969367323948}

scikit-learn pruning path (first entries):
  alpha=0, impurity=-6.1711e-14
  alpha=3.02568e-20, impurity=-6.1711e-14
  alpha=3.02568e-20, impurity=-6.17108e-14
  alpha=3.22739e-20, impurity=-6.17107e-14
  alpha=3.36187e-20, impurity=-6.17105e-14
  alpha=3.58599e-20, impurity=-6.17104e-14
  alpha=4.03424e-20, impurity=-6.17104e-14
  alpha=4.03424e-20, impurity=-6.17103e-14
  alpha=4.03424e-20, impurity=-6.17102e-14
  alpha=4.03424e-20, impurity=-6.171e-14
  (path computation time: 0.1556s)

=== Custom tree